In [1]:
using ComputableDAGs
using Pkg
Pkg.develop(; path="/home/reinha57/repos/QEDFeynman.jl/")
using QEDFeynman
using RuntimeGeneratedFunctions
using BenchmarkTools
using QEDcore, QEDprocesses
using Logging
using JLD2
using CUDA

RuntimeGeneratedFunctions.init(@__MODULE__)

MODEL = PerturbativeQED()

   Resolving package versions...
  No Changes to `~/repos/ComputableDAGs.jl/Project.toml`
  No Changes to `~/repos/ComputableDAGs.jl/Manifest.toml`


perturbative QED

In [27]:
N = 32
INSTANCE_STR = "kkkke->ke"
INSTANCE = parse_process(INSTANCE_STR, QEDModel())
g = graph(INSTANCE)

cu_inputs = CuVector([
    PhaseSpacePoint(
        INSTANCE,
        MODEL,
        PhasespaceDefinition(SphericalCoordinateSystem(), ElectronRestFrame()),
        tuple((rand(SFourMomentum) for _ in 1:number_incoming_particles(INSTANCE))...),
        tuple((rand(SFourMomentum) for _ in 1:number_outgoing_particles(INSTANCE))...),
    ) for _ in 1:N
])
cu_outputs = CuVector([0.0 for _ in 1:N])
k_unopt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

compute__f357d17e_e934_11ef_1a37_e581cd4cc058 (generic function with 1 method)

In [28]:
optimize_to_fixpoint!(ReductionOptimizer(), g)
k_opt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

compute__f447dd2c_e934_11ef_1a71_31a6c6f5efa7 (generic function with 1 method)

In [29]:
K = @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
K_opt = @cuda launch = false k_opt(cu_inputs, cu_outputs, N)

CUDA.HostKernel for compute__f447dd2c_e934_11ef_1a71_31a6c6f5efa7(CuDeviceVector{PhaseSpacePoint{ScatteringProcess{Tuple{Photon, Photon, Photon, Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, PolarizationX, PolarizationX, PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Electron, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Photon, SFourMomentum}, ParticleStateful{Outgoing, Electron, SFourMomentum}}, SFourMomentum}, 1}, CuDeviceVector{Float64, 1}, Int64)

In [30]:
K_inlined = @cuda launch = false always_inline = true k_unopt(cu_inputs, cu_outputs, N)

CUDA.HostKernel for compute__f357d17e_e934_11ef_1a37_e581cd4cc058(CuDeviceVector{PhaseSpacePoint{ScatteringProcess{Tuple{Photon, Photon, Photon, Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, PolarizationX, PolarizationX, PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Electron, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Photon, SFourMomentum}, ParticleStateful{Outgoing, Electron, SFourMomentum}}, SFourMomentum}, 1}, CuDeviceVector{Float64, 1}, Int64)

In [31]:
@info CUDA.memory(K)
@info CUDA.memory(K_opt)
@info CUDA.memory(K_inlined)

@info CUDA.registers(K)

┌ Info: (local = 210896, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sdnNjb2RlLXJlbW90ZQ==.jl:1
┌ Info: (local = 76456, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sdnNjb2RlLXJlbW90ZQ==.jl:2
┌ Info: (local = 7400, shared = 0, constant = 0)
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sdnNjb2RlLXJlbW90ZQ==.jl:3
┌ Info: 255
└ @ Main /home/reinha57/repos/ComputableDAGs.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sdnNjb2RlLXJlbW90ZQ==.jl:5


In [32]:
CUDA.@sync (@cuda threads=32 blocks=N÷32 k_opt(cu_inputs, cu_outputs, N))

CUDA.HostKernel for compute__f447dd2c_e934_11ef_1a71_31a6c6f5efa7(CuDeviceVector{PhaseSpacePoint{ScatteringProcess{Tuple{Photon, Photon, Photon, Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, PolarizationX, PolarizationX, PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}, ParticleStateful{Incoming, Electron, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Photon, SFourMomentum}, ParticleStateful{Outgoing, Electron, SFourMomentum}}, SFourMomentum}, 1}, CuDeviceVector{Float64, 1}, Int64)

In [33]:
g

Graph:
  Nodes: Total: 543, ComputeTaskQED_U: 7, ComputeTaskQED_V: 110, 
         DataTask: 275, ComputeTaskQED_S1: 30, ComputeTaskQED_S2: 120, 
         ComputeTaskQED_Sum: 1
  Edges: 885
  Total Compute Effort: 176775.25
  Total Data Transfer: 59232.0
  Total Compute Intensity: 2.9844551931388437
